# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

# Reusable helpers from lib/ (Document/Corpus containers + VectorStore wrapper)
from lib.documents import Document, Corpus
from lib.vector_db import VectorStore

In [4]:
# .env file (created from .env.example) holds:
# OPENAI_API_KEY="YOUR_KEY"
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="YOUR_KEY"
# Key values are never printed in this notebook.

In [5]:
# Load environment variables (already done in the setup cell; repeated here so the cell order of the starter still works)
load_dotenv(find_dotenv(), override=True)
print("Environment variables loaded:", all(os.getenv(k) for k in ["OPENAI_API_KEY", "OPENAI_BASE_URL", "TAVILY_API_KEY"]))

Environment variables loaded: True


### VectorDB Instance

In [6]:
# Persistent ChromaDB client: the database is stored in ./chromadb and reused in Part 2
chroma_client = chromadb.PersistentClient(path="chromadb")
chroma_client

### Collection

In [7]:
# OpenAI embedding function routed through the Vocareum endpoint (same settings are used in Part 2)
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [8]:
# get_or_create keeps the notebook re-runnable without errors
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

# Wrap the Chroma collection with the reusable VectorStore class from lib/
vector_store = VectorStore(collection)
print(f"Collection '{collection.name}' ready. Documents before indexing: {collection.count()}")

Collection 'udaplay' ready. Documents before indexing: 15


### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"
FIELDS = ["Name", "Platform", "Genre", "Publisher", "YearOfRelease", "Description"]


def game_to_text(game: dict) -> str:
    """Build a clean, labelled text document for one game (this is what gets embedded)."""
    return "\n".join(f"{field}: {game[field]}" for field in FIELDS)


corpus = Corpus()
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Use file name (like 001) as a stable ID, keep every field as metadata
    doc_id = os.path.splitext(file_name)[0]
    corpus.append(Document(id=doc_id, content=game_to_text(game), metadata=game))

# upsert (instead of add) so re-running the notebook never duplicates or fails on existing IDs
batch = corpus.to_dict()
collection.upsert(ids=batch["ids"], documents=batch["contents"], metadatas=batch["metadatas"])

print(f"Games processed: {len(corpus)}")
print(f"Documents in collection '{collection.name}': {collection.count()}")
print("\nExample document (id 006):")
print(collection.get(ids=["006"])["documents"][0])

Games processed: 15
Documents in collection 'udaplay': 15

Example document (id 006):
Name: Pokémon Gold and Silver
Platform: Game Boy Color
Genre: Role-playing
Publisher: Nintendo
YearOfRelease: 1999
Description: Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.


### Semantic search demo
Query the persistent vector database with natural-language questions. Lower distance = more similar.

In [10]:
def semantic_search(query: str, n_results: int = 3):
    """Run a semantic query against the vector store and print the top matches with distances."""
    results = vector_store.query(query_texts=[query], n_results=n_results)
    print(f"\nQuery: {query!r}")
    for rank, (meta, dist) in enumerate(zip(results["metadatas"][0], results["distances"][0]), start=1):
        print(f"  {rank}. {meta['Name']} [{meta['Platform']}, {meta['YearOfRelease']}] "
              f"- Publisher: {meta['Publisher']} | distance={dist:.4f}")
    return results


sample_queries = [
    "Pokémon Gold and Silver release",
    "first 3D Mario platformer",
    "racing simulator on PlayStation",
]
for q in sample_queries:
    semantic_search(q)


Query: 'Pokémon Gold and Silver release'
  1. Pokémon Gold and Silver [Game Boy Color, 1999] - Publisher: Nintendo | distance=0.1059
  2. Pokémon Ruby and Sapphire [Game Boy Advance, 2002] - Publisher: Nintendo | distance=0.1408
  3. Super Mario World [Super Nintendo Entertainment System (SNES), 1990] - Publisher: Nintendo | distance=0.2251



Query: 'first 3D Mario platformer'
  1. Super Mario 64 [Nintendo 64, 1996] - Publisher: Nintendo | distance=0.1129
  2. Super Mario World [Super Nintendo Entertainment System (SNES), 1990] - Publisher: Nintendo | distance=0.1344
  3. Super Smash Bros. Melee [GameCube, 2001] - Publisher: Nintendo | distance=0.1772



Query: 'racing simulator on PlayStation'
  1. Gran Turismo 5 [PlayStation 3, 2010] - Publisher: Sony Computer Entertainment | distance=0.1216
  2. Gran Turismo [PlayStation 1, 1997] - Publisher: Sony Computer Entertainment | distance=0.1223
  3. Mario Kart 8 Deluxe [Nintendo Switch, 2017] - Publisher: Nintendo | distance=0.1905


### Persistence check

In [11]:
# Verify persistence: open a NEW client on the same path and read the collection back (Part 2 relies on this)
check_client = chromadb.PersistentClient(path="chromadb")
check_collection = check_client.get_collection("udaplay", embedding_function=embedding_fn)
print(f"Reloaded collection '{check_collection.name}' from disk with {check_collection.count()} documents.")
print("The chromadb/ folder is kept on purpose: Part 2 loads it.")

Reloaded collection 'udaplay' from disk with 15 documents.
The chromadb/ folder is kept on purpose: Part 2 loads it.
